# Vivacity Full Day Cutoff 20260526

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
#!/usr/bin/env python3
"""Create Vivacity daily datasets filtered to complete days only."""

from __future__ import annotations

import json
import shutil
from pathlib import Path

import pandas as pd


BASE = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CUTOFF = pd.Timestamp("2026-05-26")
STAMP = "to_2026-05-26"

OUT = BASE / "Vivacity_full_day_cutoff_20260526"
TREATED_IN = BASE / "Vivacity_data_cleaned"
CONTROL_IN = BASE / "Vivacity_control_data_cleaned"


def filter_csv(src: Path, dest: Path) -> dict:
    df = pd.read_csv(src, low_memory=False)
    if "date" not in df.columns:
        raise ValueError(f"{src} has no date column")

    dates = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    bad_dates = int(dates.isna().sum())
    before_rows = int(len(df))
    before_min = None if before_rows == 0 else str(dates.min().date())
    before_max = None if before_rows == 0 else str(dates.max().date())

    filtered = df.loc[dates <= CUTOFF].copy()
    filtered_dates = pd.to_datetime(filtered["date"], format="mixed", errors="coerce")
    after_rows = int(len(filtered))
    after_min = None if after_rows == 0 else str(filtered_dates.min().date())
    after_max = None if after_rows == 0 else str(filtered_dates.max().date())

    dest.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_csv(dest, index=False)

    return {
        "source_file": str(src),
        "output_file": str(dest),
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_removed": before_rows - after_rows,
        "date_min_before": before_min,
        "date_max_before": before_max,
        "date_min_after": after_min,
        "date_max_after": after_max,
        "bad_date_rows": bad_dates,
    }


def daily_files(root: Path) -> list[Path]:
    return sorted(
        p
        for p in root.rglob("*.csv")
        if p.name.endswith("_daily.csv")
        or p.name in {
            "vivacity_control_daily_merged.csv",
            "vivacity_treated_and_control_daily_merged.csv",
        }
    )


def main() -> int:
    if OUT.exists():
        shutil.rmtree(OUT)
    OUT.mkdir(parents=True, exist_ok=True)

    records = []
    for src in daily_files(TREATED_IN):
        rel = src.relative_to(TREATED_IN)
        records.append(filter_csv(src, OUT / "treated" / rel))

    for src in daily_files(CONTROL_IN):
        rel = src.relative_to(CONTROL_IN)
        records.append(filter_csv(src, OUT / "control" / rel))

    # Context-enriched daily tables are useful for later modelling, so filter them too if present.
    context_files = [
        BASE / "context_data/processed/sensors/vivacity_daily_lsoa2021_context_analysis.csv",
        BASE / "context_data/processed/sensors/vivacity_daily_lsoa2021_context_with_ltp3_index.csv",
    ]
    for src in context_files:
        if src.exists():
            records.append(filter_csv(src, OUT / "context" / f"{src.stem}_{STAMP}.csv"))

    summary = pd.DataFrame(records)
    summary_path = OUT / "vivacity_full_day_cutoff_summary.csv"
    summary.to_csv(summary_path, index=False)

    qa_dir = OUT / "quality_checks"
    qa_dir.mkdir(parents=True, exist_ok=True)
    stacked_path = OUT / "control" / "vivacity_treated_and_control_daily_merged.csv"
    if stacked_path.exists():
        stacked = pd.read_csv(stacked_path, low_memory=False)
        stacked_dates = pd.to_datetime(stacked["date"], format="mixed", errors="coerce")
        stacked = stacked.assign(_date_parsed=stacked_dates)
        duplicate_keys = (
            stacked.groupby(["dataset_role", "analysis_scheme_id", "countline_id", "_date_parsed"], dropna=False)
            .size()
            .reset_index(name="rows_per_key")
            .query("rows_per_key > 1")
            .rename(columns={"_date_parsed": "date"})
        )
        duplicate_keys.to_csv(qa_dir / "vivacity_full_day_duplicate_dates_check.csv", index=False)

        coverage = (
            stacked.groupby(["dataset_role", "analysis_scheme_id", "countline_id", "countline_name"], dropna=False)
            .agg(
                first_date=("_date_parsed", "min"),
                last_date=("_date_parsed", "max"),
                daily_rows=("_date_parsed", "size"),
                bad_date_rows=("_date_parsed", lambda s: int(s.isna().sum())),
                active_travel_total=("active_travel_total", "sum"),
            )
            .reset_index()
        )
        coverage.to_csv(qa_dir / "vivacity_full_day_countline_coverage.csv", index=False)

    grouped = {
        "cutoff_inclusive": str(CUTOFF.date()),
        "files_filtered": int(len(summary)),
        "rows_before_total": int(summary["rows_before"].sum()),
        "rows_after_total": int(summary["rows_after"].sum()),
        "rows_removed_total": int(summary["rows_removed"].sum()),
        "max_output_date": str(summary["date_max_after"].max()),
        "summary_csv": str(summary_path),
        "coverage_csv": str(qa_dir / "vivacity_full_day_countline_coverage.csv"),
        "duplicate_dates_csv": str(qa_dir / "vivacity_full_day_duplicate_dates_check.csv"),
        "output_root": str(OUT),
    }
    (OUT / "vivacity_full_day_cutoff_summary.json").write_text(
        json.dumps(grouped, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(grouped, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
